# Дедупликация BigVul (шаг 0, финал)

**Вход:** `data/processed/dedup_input_clean.csv` (5375 строк)
**Выход:** `data/processed/dedup_final.csv` — по одной строке на кластер

Этапы:
1. Сильная нормализация: идентификаторы -> v0,v1,.., числа -> N, строки -> S
   (rename-only дубликаты схлопываются в точные совпадения)
2. MinHash (100 пермьютов, LSH 25x4) + точная верификация Jaccard >= 0.7
3. Кластер: одна метка CWE (большинство), один сплит (большинство, ничья -> train)
4. Отсечение классов с <15 примерами (после дедупа)

Чистый numpy, ~5.4k функций -> 1-3 минуты.

In [ ]:
import re
import numpy as np
import pandas as pd

C_KEYWORDS = set("""auto break case char const continue default do double else enum extern
float for goto if inline int long register restrict return short signed sizeof static
struct switch typedef union unsigned void volatile while _Bool _Complex _Imaginary
bool true false NULL""".split())

TOKEN_RE = re.compile(r"""
    (?P<ident>[A-Za-z_][A-Za-z0-9_]*)
  | (?P<number>\d+(\.\d+)?)
  | (?P<string>"(?:[^"\\]|\\.)*"|'(?:[^'\\]|\\.)*')
  | (?P<punct>\S)
""", re.X)

def strong_norm(code: str) -> list:
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.S)
    code = re.sub(r"//[^\n]*", " ", code)
    out, counter = [], {}
    for m in TOKEN_RE.finditer(code):
        k = m.lastgroup
        if k == "ident":
            w = m.group()
            if w in C_KEYWORDS:
                out.append(w)
            else:
                if w not in counter:
                    counter[w] = f"v{len(counter)}"
                out.append(counter[w])
        elif k == "number":
            out.append("N")
        else:
            out.append("S" if k == "string" else m.group())
    return out

def shingles(tokens, n=5):
    if len(tokens) < n:
        return {hash(tuple(tokens))}
    return {hash(tuple(tokens[i:i+n])) for i in range(len(tokens)-n+1)}

def jac(x, y):
    return len(x & y) / len(x | y)

def minhash_sig(sh, n_perm=100, seed=42):
    rng = np.random.RandomState(seed)
    A = rng.randint(1, 2**31 - 1, n_perm)
    B = rng.randint(0, 2**31 - 1, n_perm)
    P = 2**31 - 1
    arr = np.array(sorted(sh), dtype=np.int64)
    vals = (arr[:, None] * A[None, :] + B[None, :]) % P
    return vals.min(axis=0)

def cluster_docs(sigs, shingle_sets, n_perm=100, band_rows=4, thr=0.7):
    n_bands = n_perm // band_rows
    parent = list(range(len(sigs)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for b in range(n_bands):
        buckets = {}
        for i, s in enumerate(sigs):
            buckets.setdefault(hash(tuple(s[b*band_rows:(b+1)*band_rows])), []).append(i)
        for ids in buckets.values():
            for j in ids[1:]:
                union(ids[0], j)
    groups = {}
    for i in range(len(sigs)):
        groups.setdefault(find(i), []).append(i)
    clusters = []
    for ids in groups.values():
        if len(ids) == 1:
            clusters.append(ids)
            continue
        keep = [ids[0]]
        for x in ids[1:]:
            if any(jac(shingle_sets[x], shingle_sets[y]) >= thr for y in keep):
                keep.append(x)
            else:
                clusters.append([x])
        clusters.append(keep)
    return clusters

## 1. Загрузка и кластеризация

In [ ]:
df = pd.read_csv("data/processed/dedup_input_clean.csv")
print(f"строк на входе: {len(df)}")

toks = df["func_before"].map(strong_norm)
shs  = toks.map(shingles)

# точные дубликаты после сильной нормализации
exact_groups = df.assign(_k=shs.map(lambda s: hash(tuple(sorted(s))))).groupby("_k").size()
print(f"точных дубликатов (rename-insensitive): {int((exact_groups - 1).clip(lower=0).sum())}")

sigs = [minhash_sig(s) for s in shs]
clusters = cluster_docs(sigs, list(shs))
sizes = sorted((len(c) for c in clusters), reverse=True)
print(f"кластеров: {len(clusters)} | размеры топ-5: {sizes[:5]} | синглтонов: {sum(1 for s in sizes if s == 1)}")

## 2. Глазами: примеры near-dup пар

In [ ]:
rows = []
for ci, ids in enumerate(clusters):
    sub = df.iloc[list(ids)]
    cwe_votes = sub["cwe_id"].value_counts()
    cwe_final = int(cwe_votes.idxmax())
    conflict = len(cwe_votes) > 1
    split_votes = sub["split"].value_counts()
    if len(split_votes) == 1:
        split_new = split_votes.index[0]           # единогласный кластер (в т.ч. синглтон) -> свой сплит
    else:
        split_new = split_votes.idxmax() if split_votes.max() > split_votes.min() else "train"
    cands = sub[sub["cwe_id"] == cwe_final]
    rep = cands.loc[cands["func_before"].str.len().idxmax()]
    rows.append({"cluster_id": ci, "rep_idx": int(rep.name), "cwe_final": cwe_final,
                 "cluster_size": len(ids), "conflict": conflict, "split_new": split_new})

cl = pd.DataFrame(rows)
print(f"кластеров с конфликтом меток: {int(cl['conflict'].sum())} / {len(cl)}")

# экспорт конфликтных кластеров на разбор
conf_rows = []
for ci in cl.loc[cl["conflict"], "cluster_id"]:
    sub = df.iloc[list(clusters[ci])]
    for _, r in sub.iterrows():
        conf_rows.append({"cluster_id": ci, "cwe": int(r["cwe_id"]), "cve": r["cve"],
                          "project": r["project"], "size": len(clusters[ci])})
pd.DataFrame(conf_rows).to_csv("data/processed/dedup_conflicts.csv", index=False)
print("конфликты экспортированы: data/processed/dedup_conflicts.csv")

out = df.loc[cl["rep_idx"]].reset_index(drop=True).drop(columns=["norm", "hash"], errors="ignore")
out["cwe_id"] = cl["cwe_final"].values
out["cluster_id"] = cl["cluster_id"].values
out["cluster_size"] = cl["cluster_size"].values
out["conflict"] = cl["conflict"].values
out["split"] = cl["split_new"].values

leak = out.groupby("cluster_id")["split"].nunique()
print(f"утечка после починки: {int((leak > 1).sum())} (должно быть 0)")
print("сплиты:", out["split"].value_counts().to_dict())

## 3. Метки кластеров и починка сплитов

## 4. Отсечение мелких классов и конфликтных кластеров

Конфликтные кластеры (разные метки CWE у членов) по умолчанию выкидываем: голосование большинством при 119 = 40% данных систематически смещает такие кластеры в доминирующий класс. Потеря ~2% данных чище этого смещения. Список на ручной разбор: `data/processed/dedup_conflicts.csv`.

In [ ]:
DROP_CONFLICTS = True   # False — оставить с меткой большинства

n0 = len(out)
if DROP_CONFLICTS:
    out = out[~out["conflict"]]
    print(f"убрано конфликтных кластеров: {n0 - len(out)} строк")

vc = out["cwe_id"].value_counts()
keep = set(vc[vc >= 15].index)
final = out[out["cwe_id"].isin(keep)].copy()

print(f"классов: {len(keep)} (было {len(vc)})")
print(f"строк: {len(final)} (на входе шага было {n0})")
print("\nфинальное распределение:")
print(final["cwe_id"].value_counts().to_string())
print("\nсплиты:", final["split"].value_counts().to_dict())

final.to_csv("data/processed/dedup_final.csv", index=False)
print("\nsaved: data/processed/dedup_final.csv")

In [ ]:
vc = out["cwe_id"].value_counts()
keep = set(vc[vc >= 15].index)
final = out[out["cwe_id"].isin(keep)].copy()

print(f"классов осталось: {len(keep)} (было {len(vc)})")
print(f"строк: {len(final)} (было {len(out)})")
print("\nфинальное распределение:")
print(final["cwe_id"].value_counts().to_string())

final.to_csv("data/processed/dedup_final.csv", index=False)
print("\nsaved: data/processed/dedup_final.csv")